# Spectral clustering

_Classical ML_

**Cluster by connectivity, not distance — cut the graph at its thin seams.**

Two crescents can interleave so that points across the gap are closer than points along the same arc.

     k-means, which judges by raw distance, fails badly here.

---

This notebook is a practice scaffold for the **Spectral clustering** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_wine

data = load_wine(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — scikit-learn

In [ ]:
import numpy as np
from sklearn.datasets import make_moons
from sklearn.cluster import SpectralClustering, KMeans
from sklearn.metrics import adjusted_rand_score

X, y = make_moons(n_samples=300, noise=0.06, random_state=0)

sc = SpectralClustering(n_clusters=2, affinity="nearest_neighbors",
                        n_neighbors=10, assign_labels="kmeans",
                        random_state=0)
sc_labels = sc.fit_predict(X)

km_labels = KMeans(n_clusters=2, n_init=10, random_state=0).fit_predict(X)

print("spectral cluster sizes:", np.bincount(sc_labels))
print("spectral ARI vs truth:", round(adjusted_rand_score(y, sc_labels), 3))
print("k-means  ARI vs truth:", round(adjusted_rand_score(y, km_labels), 3))
print("-> spectral follows the curved moons; k-means does not")

## Visualize the data & results

_On the real Wine cultivars, does spectral clustering recover the same groups as k-means?_

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import SpectralClustering, KMeans
from sklearn.metrics import adjusted_rand_score

wine = load_wine()
Xs = StandardScaler().fit_transform(wine.data)
X2 = PCA(n_components=2, random_state=0).fit_transform(Xs)

sc = SpectralClustering(n_clusters=3, affinity="nearest_neighbors",
                        n_neighbors=10, random_state=0).fit_predict(X2)
km = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(X2)
print(adjusted_rand_score(wine.target, sc),
      adjusted_rand_score(wine.target, km))

colors = np.array(["#4ea1ff", "#7ee787", "#c89bff"])
fig, (ax1, ax2) = plt.subplots(1, 2, sharex=True, sharey=True)
ax1.scatter(X2[:, 0], X2[:, 1], c=colors[sc], s=18)
ax1.set_title("Spectral clustering on Wine")
ax2.scatter(X2[:, 0], X2[:, 1], c=colors[km], s=18)
ax2.set_title("k-means on Wine")
for ax in (ax1, ax2):
    ax.set_xlabel("PCA component 1")
    ax.set_ylabel("PCA component 2")
plt.show()